# ViLT + CLIP Cross-Attention Fusion — Fake News Detection

## Dual Vision-Language Architecture with Metadata

**Key Features:**
- ✅ **Dual Models**: ViLT (joint V+L) + CLIP (dual V+L)
- ✅ **Cross-Attention Fusion**: Inter-model attention mechanisms
- ✅ **Metadata Integration**: Source domain, engagement, text stats
- ✅ **TF-IDF Ensemble**: Neural + classical ML blending
- ✅ **Kaggle-Ready**: Checkpointing, efficient GPU usage

**Architecture:**
```
          ViLT Encoder              CLIP Encoders
               ↓                          ↓
        [Joint V+T Embed]         [Image Embed | Text Embed]
               ↓                          ↓
        ─────────────────────────────────────────
        │  Cross-Attention Fusion Network      │
        │  - ViLT ↔ CLIP-Image                 │
        │  - ViLT ↔ CLIP-Text                  │
        │  - CLIP-Image ↔ CLIP-Text            │
        │  + Metadata Conditioning              │
        ─────────────────────────────────────────
                       ↓
              Deep Classifier → [Real | Fake]
```

In [ ]:
# Install dependencies
!pip install -q transformers datasets pillow scikit-learn tqdm

In [ ]:
"""
ViLT + CLIP Dual-Encoder Fusion — FakeNewsNet Binary Classification
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Architecture:
  - ViLT: dandelin/vilt-b32-mlm (Joint vision-language encoder)
  - CLIP: openai/clip-vit-base-patch32 (Dual vision+text encoders)
  - Cross-Attention: Multi-head attention between all encoder outputs
  - Metadata: Source domain, tweet engagement, text statistics
  - Classifier: Deep MLP with residual connections

Dataset: rickstello/FakeNewsNet (train split)
Label: 0 = real, 1 = fake
Split: 80% train / 10% val / 10% test (stratified)
CV: 5-fold on train pool
Target: ≥89% accuracy | Balanced metrics
"""

from __future__ import annotations

import csv
import io
import os
import random
import math
import warnings
from collections import Counter
from contextlib import nullcontext
from dataclasses import dataclass
from typing import Any

import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    ViltProcessor, ViltModel,
    CLIPProcessor, CLIPModel,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════════════════
@dataclass
class Config:
    # Models
    vilt_model_name: str = "dandelin/vilt-b32-mlm"
    clip_model_name: str = "openai/clip-vit-base-patch32"
    dataset_name: str = "rickstello/FakeNewsNet"
    dataset_split: str = "train"
    max_samples: int | None = None
    num_classes: int = 2

    # Training
    batch_size: int = 12  # Reduced for dual model
    epochs: int = 8
    lr_head: float = 3e-4
    lr_backbone: float = 2e-6  # Lower for dual model
    weight_decay: float = 0.05
    grad_accumulation: int = 3
    max_grad_norm: float = 1.0
    warmup_ratio: float = 0.15

    # Staged fine-tuning
    freeze_epochs: int = 2
    unfreeze_last_layers: int = 2

    # Data
    n_splits: int = 5
    num_workers: int = 2
    image_timeout_s: float = 10.0
    vilt_image_size: int = 384
    clip_image_size: int = 224
    text_max_length: int = 64
    train_frac: float = 0.8
    val_frac: float = 0.1
    test_frac: float = 0.1
    require_image_source: bool = False

    # Model architecture
    vilt_embed_dim: int | None = None
    clip_embed_dim: int | None = None
    fusion_dim: int = 256
    n_attn_heads: int = 4
    head_dropout: float = 0.40
    label_smoothing: float = 0.05
    
    # Metadata
    use_metadata: bool = True
    metadata_dim: int = 64
    max_domains: int = 500
    
    # TF-IDF ensemble
    use_tfidf_ensemble: bool = True
    tfidf_max_features: int = 80_000
    tfidf_min_df: int = 2
    tfidf_ngram_max: int = 3

    # Early stopping
    early_stopping_patience: int = 3

    # I/O (Kaggle paths)
    results_dir: str = "/kaggle/working"
    checkpoint_dir: str = "/kaggle/working/checkpoints"
    results_file_prefix: str = "vilt_clip_fusion"
    save_checkpoints: bool = True
    force_cpu: bool = False

    # AMP
    use_amp: bool = True


config = Config()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  DEVICE / AMP
# ══════════════════════════════════════════════════════════════════════
def resolve_device(*, force_cpu: bool = False) -> torch.device:
    if force_cpu or not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        major, _ = torch.cuda.get_device_capability(0)
        if major < 7:
            print(f"GPU compute capability {major} < 7; using CPU.")
            return torch.device("cpu")
    except Exception as e:
        print(f"CUDA check failed: {e}; using CPU.")
        return torch.device("cpu")
    return torch.device("cuda")


def build_amp_helpers(use_cuda: bool):
    if not use_cuda:
        return (lambda: nullcontext()), torch.cuda.amp.GradScaler(enabled=False)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        return (lambda: torch.amp.autocast("cuda")), scaler
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=True)
        return (lambda: torch.cuda.amp.autocast()), scaler

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  TEXT / IMAGE / METADATA HELPERS
# ══════════════════════════════════════════════════════════════════════
def _clean(v: object) -> str:
    s = str(v).strip() if v is not None else ""
    return "" if not s or s.lower() in {"null", "none", "nan"} else s


def get_article_text(ex: dict) -> str:
    parts, seen = [], set()
    for k in ("title", "headline", "text", "article", "content", "body"):
        s = _clean(ex.get(k))
        norm = " ".join(s.lower().split())
        if s and norm not in seen:
            parts.append(s)
            seen.add(norm)
    return " ".join(parts)


def extract_metadata_features(ex: dict) -> dict:
    domain = _clean(ex.get("source_domain", "")).lower() or "unknown"
    
    try:
        tweet_num = float(ex.get("tweet_num", 0) or 0)
    except (TypeError, ValueError):
        tweet_num = 0.0
    
    tweet_bin = 0 if tweet_num <= 0 else min(9, int(math.log1p(tweet_num)))
    
    text = get_article_text(ex)
    text_length = len(text.split())
    text_len_bin = min(9, int(math.log1p(text_length)))
    
    has_image = get_image_url(ex) is not None or decode_hf_image(ex) is not None
    
    return {
        "domain": domain,
        "tweet_bin": tweet_bin,
        "text_len_bin": text_len_bin,
        "has_image": int(has_image),
        "tweet_num_log": math.log1p(tweet_num),
    }


def get_metadata_text(ex: dict) -> str:
    """For TF-IDF ensemble."""
    text = get_article_text(ex)
    domain = _clean(ex.get("source_domain"))
    try:
        tweets = float(ex.get("tweet_num", 0) or 0)
    except (TypeError, ValueError):
        tweets = 0.0
    tweet_bucket = f"tweet_bin_{min(9, int(math.log1p(tweets)))}" if tweets > 0 else "tweet_bin_0"
    parts = [text]
    if domain:
        parts.extend([f"domain_{domain.lower()}", f"domain_tokens_{domain.lower().replace('.', ' ')}"])
    parts.append(tweet_bucket)
    return " ".join(parts)


def get_image_url(ex: dict) -> str | None:
    for k in ("image_url", "top_img_url", "top_img", "image", "img_url", "tweet_image_url", "thumbnail"):
        v = ex.get(k)
        if v and isinstance(v, str) and v.strip() and v.lower() not in ("null", "none", ""):
            return v.strip()
    return None


def decode_hf_image(ex: dict) -> Image.Image | None:
    v = ex.get("image")
    if v is None:
        return None
    if isinstance(v, Image.Image):
        return v.convert("RGB")
    if isinstance(v, dict) and v.get("bytes"):
        try:
            return Image.open(io.BytesIO(v["bytes"])).convert("RGB")
        except Exception:
            return None
    return None


def load_rgb_image(ex: dict, target_size: int = 224) -> Image.Image:
    url = get_image_url(ex)
    if url:
        try:
            r = requests.get(url, timeout=config.image_timeout_s, stream=True)
            r.raise_for_status()
            return Image.open(io.BytesIO(r.content)).convert("RGB")
        except Exception:
            pass
    pil = decode_hf_image(ex)
    if pil is not None:
        return pil
    return Image.new("RGB", (target_size, target_size), color=(128, 128, 128))


def get_binary_label_fake_is_one(ex: dict) -> int | None:
    if ex.get("real") is not None:
        try:
            r = int(ex["real"])
            if r in (0, 1):
                return 1 - r
        except (TypeError, ValueError):
            pass
    for k in ("is_fake", "label"):
        v = ex.get(k)
        if v is not None:
            try:
                lab = int(v)
                if lab in (0, 1):
                    return lab
            except (TypeError, ValueError):
                pass
    return None


def row_has_image_source(ex: dict) -> bool:
    return get_image_url(ex) is not None or decode_hf_image(ex) is not None

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  METADATA ENCODER
# ══════════════════════════════════════════════════════════════════════
class MetadataEncoder:
    def __init__(self, max_domains: int = 500):
        self.max_domains = max_domains
        self.domain_vocab = {}
        self.fitted = False
    
    def fit(self, hf_dataset, indices: list[int]):
        domain_counts = Counter()
        for i in indices:
            meta = extract_metadata_features(hf_dataset[i])
            domain_counts[meta["domain"]] += 1
        
        top_domains = [d for d, _ in domain_counts.most_common(self.max_domains - 1)]
        self.domain_vocab = {d: i for i, d in enumerate(top_domains)}
        self.domain_vocab["<UNK>"] = len(self.domain_vocab)
        self.fitted = True
        print(f"  Metadata encoder: {len(self.domain_vocab)} unique domains")
    
    def encode(self, meta_dict: dict) -> dict:
        if not self.fitted:
            raise RuntimeError("MetadataEncoder must be fitted first")
        
        domain_id = self.domain_vocab.get(meta_dict["domain"], self.domain_vocab["<UNK>"])
        
        return {
            "domain_id": domain_id,
            "tweet_bin": meta_dict["tweet_bin"],
            "text_len_bin": meta_dict["text_len_bin"],
            "has_image": meta_dict["has_image"],
            "tweet_num_log": meta_dict["tweet_num_log"],
        }
    
    @property
    def num_domains(self) -> int:
        return len(self.domain_vocab) if self.fitted else 0

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  DATASET
# ══════════════════════════════════════════════════════════════════════
class DualEncoderDataset(Dataset):
    """
    Returns outputs for both ViLT and CLIP processors + metadata.
    """
    def __init__(self, hf_dataset, vilt_processor, clip_processor,
                 metadata_encoder: MetadataEncoder, indices: list[int] | None = None):
        self.rows = hf_dataset
        self.vilt_processor = vilt_processor
        self.clip_processor = clip_processor
        self.metadata_encoder = metadata_encoder
        self.indices = indices if indices is not None else list(range(len(hf_dataset)))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx: int):
        i = self.indices[idx]
        ex = self.rows[i]
        text = get_article_text(ex)
        if not text:
            raise ValueError("empty text")

        lab = get_binary_label_fake_is_one(ex)
        if lab is None:
            raise ValueError("missing label")

        # Load images at different sizes
        img_vilt = load_rgb_image(ex, config.vilt_image_size)
        img_clip = load_rgb_image(ex, config.clip_image_size)

        # ViLT encoding
        vilt_enc = self.vilt_processor(
            images=img_vilt,
            text=text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=config.text_max_length,
        )
        vilt_item = {f"vilt_{k}": v.squeeze(0) for k, v in vilt_enc.items()}

        # CLIP encoding
        clip_enc = self.clip_processor(
            text=text,
            images=img_clip,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=config.text_max_length,
        )
        clip_item = {f"clip_{k}": v.squeeze(0) for k, v in clip_enc.items()}

        # Metadata
        item = {**vilt_item, **clip_item}
        if config.use_metadata:
            meta = extract_metadata_features(ex)
            encoded_meta = self.metadata_encoder.encode(meta)
            item["metadata"] = {
                "domain_id": torch.tensor(encoded_meta["domain_id"], dtype=torch.long),
                "tweet_bin": torch.tensor(encoded_meta["tweet_bin"], dtype=torch.long),
                "text_len_bin": torch.tensor(encoded_meta["text_len_bin"], dtype=torch.long),
                "has_image": torch.tensor(encoded_meta["has_image"], dtype=torch.float32),
                "tweet_num_log": torch.tensor(encoded_meta["tweet_num_log"], dtype=torch.float32),
            }

        item["labels"] = torch.tensor(lab, dtype=torch.long)
        return item


def collate_fn(batch):
    labels = torch.stack([b.pop("labels") for b in batch], dim=0)
    
    # Collect metadata if present
    metadata = None
    if "metadata" in batch[0]:
        metadata = {
            k: torch.stack([b["metadata"][k] for b in batch], dim=0)
            for k in batch[0]["metadata"].keys()
        }
        for b in batch:
            b.pop("metadata")
    
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0].keys()}
    out["labels"] = labels
    if metadata is not None:
        out["metadata"] = metadata
    return out

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CROSS-ATTENTION & METADATA MODULES
# ══════════════════════════════════════════════════════════════════════
class CrossAttentionBlock(nn.Module):
    def __init__(self, D: int, n_heads: int, dropout: float = 0.10):
        super().__init__()
        self.attn = nn.MultiheadAttention(D, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(D)
        self.drop = nn.Dropout(dropout)
        self.ff = nn.Sequential(
            nn.Linear(D, D * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(D * 4, D)
        )
        self.norm2 = nn.LayerNorm(D)

    def forward(self, query: torch.Tensor, key_value: torch.Tensor) -> torch.Tensor:
        q = query.unsqueeze(1)
        kv = key_value.unsqueeze(1)
        attn_out, _ = self.attn(q, kv, kv)
        x = self.norm(query + self.drop(attn_out.squeeze(1)))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


class MetadataEmbedding(nn.Module):
    def __init__(self, num_domains: int, metadata_dim: int = 64):
        super().__init__()
        self.metadata_dim = metadata_dim
        
        self.domain_embed = nn.Embedding(num_domains, metadata_dim // 2)
        self.tweet_bin_embed = nn.Embedding(10, 16)
        self.text_len_bin_embed = nn.Embedding(10, 16)
        
        total_embed_dim = (metadata_dim // 2) + 16 + 16 + 2
        self.projection = nn.Sequential(
            nn.Linear(total_embed_dim, metadata_dim),
            nn.LayerNorm(metadata_dim),
            nn.GELU(),
            nn.Dropout(0.1)
        )
    
    def forward(self, metadata: dict) -> torch.Tensor:
        domain_emb = self.domain_embed(metadata["domain_id"])
        tweet_bin_emb = self.tweet_bin_embed(metadata["tweet_bin"])
        text_len_emb = self.text_len_bin_embed(metadata["text_len_bin"])
        
        has_image = metadata["has_image"].unsqueeze(-1)
        tweet_log = metadata["tweet_num_log"].unsqueeze(-1)
        
        features = torch.cat([
            domain_emb, tweet_bin_emb, text_len_emb, has_image, tweet_log
        ], dim=-1)
        
        return self.projection(features)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  MAIN MODEL: ViLT + CLIP Dual-Encoder Fusion
# ══════════════════════════════════════════════════════════════════════
class ViLTCLIPDualFusion(nn.Module):
    """
    Combines ViLT (joint V+L) and CLIP (dual V+L) with cross-attention.
    
    Encoders:
      - ViLT: Joint vision-language embedding
      - CLIP: Separate image and text embeddings
      
    Fusion:
      - Cross-attention between all encoder outputs
      - Metadata conditioning
      - Deep classifier with residual connections
    """
    def __init__(self, num_domains: int, num_classes: int = 2):
        super().__init__()
        
        print(f"  Loading ViLT: {config.vilt_model_name}")
        self.vilt = ViltModel.from_pretrained(config.vilt_model_name)
        
        print(f"  Loading CLIP: {config.clip_model_name}")
        self.clip = CLIPModel.from_pretrained(config.clip_model_name)

        # Detect embedding dimensions
        D = config.fusion_dim
        self.vilt_dim = self.vilt.config.hidden_size
        self.clip_dim = self.clip.config.projection_dim
        config.vilt_embed_dim = self.vilt_dim
        config.clip_embed_dim = self.clip_dim
        drop = config.head_dropout
        
        print(f"    ViLT embedding dim: {self.vilt_dim}")
        print(f"    CLIP embedding dim: {self.clip_dim}")
        print(f"    Metadata enabled: {config.use_metadata}")

        # Project all encoders to fusion space
        self.vilt_proj = nn.Sequential(
            nn.Linear(self.vilt_dim, D),
            nn.LayerNorm(D),
            nn.GELU(),
            nn.Dropout(0.10)
        )
        self.clip_img_proj = nn.Sequential(
            nn.Linear(self.clip_dim, D),
            nn.LayerNorm(D),
            nn.GELU(),
            nn.Dropout(0.10)
        )
        self.clip_txt_proj = nn.Sequential(
            nn.Linear(self.clip_dim, D),
            nn.LayerNorm(D),
            nn.GELU(),
            nn.Dropout(0.10)
        )

        # Metadata
        if config.use_metadata:
            self.metadata_embed = MetadataEmbedding(num_domains, config.metadata_dim)
            self.meta_proj = nn.Sequential(
                nn.Linear(config.metadata_dim, D),
                nn.LayerNorm(D),
                nn.GELU(),
                nn.Dropout(0.10)
            )

        # Cross-attention: ViLT ↔ CLIP components
        self.ca_vilt2clip_img = CrossAttentionBlock(D, config.n_attn_heads, dropout=0.10)
        self.ca_vilt2clip_txt = CrossAttentionBlock(D, config.n_attn_heads, dropout=0.10)
        self.ca_clip_img2vilt = CrossAttentionBlock(D, config.n_attn_heads, dropout=0.10)
        self.ca_clip_txt2vilt = CrossAttentionBlock(D, config.n_attn_heads, dropout=0.10)
        
        # CLIP internal cross-attention
        self.ca_clip_img2txt = CrossAttentionBlock(D, config.n_attn_heads, dropout=0.10)
        self.ca_clip_txt2img = CrossAttentionBlock(D, config.n_attn_heads, dropout=0.10)
        
        if config.use_metadata:
            self.ca_meta2vilt = CrossAttentionBlock(D, config.n_attn_heads, dropout=0.10)
            self.ca_meta2clip_img = CrossAttentionBlock(D, config.n_attn_heads, dropout=0.10)
            self.ca_meta2clip_txt = CrossAttentionBlock(D, config.n_attn_heads, dropout=0.10)

        # Classifier
        # Without metadata: [vilt, clip_img, clip_txt, 6 cross-attn] = 9*D
        # With metadata: [vilt, clip_img, clip_txt, meta, 9 cross-attn] = 13*D
        fused_dim = 13 * D if config.use_metadata else 9 * D
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(drop),
            nn.Linear(fused_dim, 768),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(drop * 0.6),
            nn.Linear(256, num_classes),
        )
        
        # Xavier init
        for m in self.modules():
            if isinstance(m, nn.Linear) and m not in list(self.vilt.modules()) + list(self.clip.modules()):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def freeze_backbones(self):
        for p in self.vilt.parameters():
            p.requires_grad = False
        for p in self.clip.parameters():
            p.requires_grad = False

    def unfreeze_last_layers(self, n: int = 2):
        self.freeze_backbones()
        
        # Unfreeze ViLT last layers
        try:
            vilt_layers = self.vilt.encoder.layer
            for layer in vilt_layers[-n:]:
                for p in layer.parameters():
                    p.requires_grad = True
            for p in self.vilt.layernorm.parameters():
                p.requires_grad = True
        except AttributeError:
            for p in self.vilt.parameters():
                p.requires_grad = True

        # Unfreeze CLIP last layers
        try:
            # CLIP vision encoder
            vis_layers = self.clip.vision_model.encoder.layers
            for layer in vis_layers[-n:]:
                for p in layer.parameters():
                    p.requires_grad = True
            for p in self.clip.vision_model.post_layernorm.parameters():
                p.requires_grad = True
        except AttributeError:
            pass

        try:
            # CLIP text encoder
            txt_layers = self.clip.text_model.encoder.layers
            for layer in txt_layers[-n:]:
                for p in layer.parameters():
                    p.requires_grad = True
            for p in self.clip.text_model.final_layer_norm.parameters():
                p.requires_grad = True
        except AttributeError:
            pass

        n_vilt = sum(p.numel() for p in self.vilt.parameters() if p.requires_grad)
        n_clip = sum(p.numel() for p in self.clip.parameters() if p.requires_grad)
        print(f"    ViLT unfrozen: {n_vilt/1e6:.1f}M params")
        print(f"    CLIP unfrozen: {n_clip/1e6:.1f}M params")

    def backbone_params_for_optimizer(self, lr: float) -> list[dict]:
        vilt_params = [p for p in self.vilt.parameters() if p.requires_grad]
        clip_params = [p for p in self.clip.parameters() if p.requires_grad]
        return [{
            "params": vilt_params + clip_params,
            "lr": lr,
            "weight_decay": config.weight_decay
        }]

    def head_params_for_optimizer(self, lr: float) -> list[dict]:
        head_modules = [
            self.vilt_proj, self.clip_img_proj, self.clip_txt_proj,
            self.ca_vilt2clip_img, self.ca_vilt2clip_txt,
            self.ca_clip_img2vilt, self.ca_clip_txt2vilt,
            self.ca_clip_img2txt, self.ca_clip_txt2img,
            self.classifier
        ]
        if config.use_metadata:
            head_modules.extend([
                self.metadata_embed, self.meta_proj,
                self.ca_meta2vilt, self.ca_meta2clip_img, self.ca_meta2clip_txt
            ])
        return [{
            "params": [p for m in head_modules for p in m.parameters()],
            "lr": lr,
            "weight_decay": config.weight_decay
        }]

    def forward(
        self,
        vilt_input_ids: torch.Tensor,
        vilt_attention_mask: torch.Tensor,
        vilt_pixel_values: torch.Tensor,
        vilt_pixel_mask: torch.Tensor,
        clip_input_ids: torch.Tensor,
        clip_pixel_values: torch.Tensor,
        clip_attention_mask: torch.Tensor,
        metadata: dict | None = None,
        labels: torch.Tensor | None = None,
        class_weights: torch.Tensor | None = None,
        **unused,
    ):
        # ViLT encoding (joint vision-language)
        vilt_out = self.vilt(
            input_ids=vilt_input_ids,
            attention_mask=vilt_attention_mask,
            pixel_values=vilt_pixel_values,
            pixel_mask=vilt_pixel_mask,
            return_dict=True
        )
        vilt_emb = vilt_out.pooler_output  # (B, vilt_dim)

        # CLIP encoding (separate image and text)
        clip_out = self.clip(
            input_ids=clip_input_ids,
            pixel_values=clip_pixel_values,
            attention_mask=clip_attention_mask,
            return_dict=True
        )
        
        clip_img_emb = clip_out.image_embeds  # (B, clip_dim)
        clip_txt_emb = clip_out.text_embeds   # (B, clip_dim)

        # Project to fusion space
        p_vilt = self.vilt_proj(vilt_emb)
        p_clip_img = self.clip_img_proj(clip_img_emb)
        p_clip_txt = self.clip_txt_proj(clip_txt_emb)

        # Cross-attention fusion
        # ViLT ↔ CLIP
        ca_v2ci = self.ca_vilt2clip_img(p_vilt, p_clip_img)
        ca_v2ct = self.ca_vilt2clip_txt(p_vilt, p_clip_txt)
        ca_ci2v = self.ca_clip_img2vilt(p_clip_img, p_vilt)
        ca_ct2v = self.ca_clip_txt2vilt(p_clip_txt, p_vilt)
        
        # CLIP internal
        ca_ci2ct = self.ca_clip_img2txt(p_clip_img, p_clip_txt)
        ca_ct2ci = self.ca_clip_txt2img(p_clip_txt, p_clip_img)

        if config.use_metadata and metadata is not None:
            meta_emb = self.metadata_embed(metadata)
            p_meta = self.meta_proj(meta_emb)
            
            ca_m2v = self.ca_meta2vilt(p_meta, p_vilt)
            ca_m2ci = self.ca_meta2clip_img(p_meta, p_clip_img)
            ca_m2ct = self.ca_meta2clip_txt(p_meta, p_clip_txt)
            
            fused = torch.cat([
                p_vilt, p_clip_img, p_clip_txt, p_meta,
                ca_v2ci, ca_v2ct, ca_ci2v, ca_ct2v, ca_ci2ct, ca_ct2ci,
                ca_m2v, ca_m2ci, ca_m2ct
            ], dim=-1)
        else:
            fused = torch.cat([
                p_vilt, p_clip_img, p_clip_txt,
                ca_v2ci, ca_v2ct, ca_ci2v, ca_ct2v, ca_ci2ct, ca_ct2ci
            ], dim=-1)

        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits, labels,
                weight=class_weights,
                label_smoothing=config.label_smoothing,
            )
        return {"loss": loss, "logits": logits}

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  METRICS & UTILITIES
# ══════════════════════════════════════════════════════════════════════
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def class_weights_from_labels(labels: np.ndarray, device: torch.device) -> torch.Tensor:
    counts = np.bincount(labels, minlength=config.num_classes).astype(np.float32)
    weights = counts.sum() / (config.num_classes * np.maximum(counts, 1.0))
    return torch.tensor(weights / weights.mean(), dtype=torch.float32, device=device)


@torch.no_grad()
def collect_probabilities(model, loader, device, autocast_ctx, class_weights=None):
    model.eval()
    all_probs, all_y = [], []
    for batch in loader:
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                 for k, v in batch.items()}
        if "metadata" in batch:
            batch["metadata"] = {k: v.to(device) for k, v in batch["metadata"].items()}
        
        with autocast_ctx():
            logits = model(**batch, labels=None)["logits"]
        probs = F.softmax(logits.float(), dim=-1)
        all_probs.append(probs.cpu().numpy())
        all_y.append(labels.cpu().numpy())
    
    return np.concatenate(all_y), np.vstack(all_probs)


def probs_to_preds(probs: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    if probs.shape[1] == 2:
        return (probs[:, 1] >= threshold).astype(np.int64)
    return probs.argmax(axis=1)


def tune_binary_blend(y_true, neural_probs, tfidf_probs=None):
    best_weight = 1.0
    best_threshold = 0.5
    best_metrics = compute_metrics(y_true, probs_to_preds(neural_probs))
    best_preds = probs_to_preds(neural_probs)
    weights = [1.0] if tfidf_probs is None else np.linspace(0.0, 1.0, 21)
    for w in weights:
        blended = neural_probs if tfidf_probs is None else w * neural_probs + (1.0 - w) * tfidf_probs
        for threshold in np.linspace(0.20, 0.80, 61):
            preds = probs_to_preds(blended, float(threshold))
            metrics = compute_metrics(y_true, preds)
            if (metrics["f1"] > best_metrics["f1"] or
                (abs(metrics["f1"] - best_metrics["f1"]) < 1e-12 and
                 metrics["accuracy"] > best_metrics["accuracy"])):
                best_weight = float(w)
                best_threshold = float(threshold)
                best_metrics = metrics
                best_preds = preds
    return best_weight, best_threshold, best_metrics, best_preds


def build_tfidf_model(hf_dataset, train_abs: list[int]):
    texts = [get_metadata_text(hf_dataset[i]) for i in train_abs]
    labels = np.array([get_binary_label_fake_is_one(hf_dataset[i]) for i in train_abs], dtype=np.int64)
    vec = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        sublinear_tf=True,
        min_df=config.tfidf_min_df,
        max_features=config.tfidf_max_features,
        ngram_range=(1, config.tfidf_ngram_max),
        token_pattern=r"(?u)\b[\w./:-]+\b",
    )
    x_train = vec.fit_transform(texts)
    clf = LogisticRegression(
        C=3.0,
        class_weight="balanced",
        max_iter=2000,
        solver="liblinear",
        random_state=SEED,
    )
    clf.fit(x_train, labels)
    train_acc = accuracy_score(labels, clf.predict(x_train))
    return vec, clf, train_acc


def tfidf_predict_proba(vec, clf, hf_dataset, indices: list[int]) -> np.ndarray:
    texts = [get_metadata_text(hf_dataset[i]) for i in indices]
    return clf.predict_proba(vec.transform(texts))


def build_valid_indices(hf_dataset, max_n: int) -> list[int]:
    valid = []
    n = min(len(hf_dataset), max_n)
    for i in tqdm(range(n), desc="Scanning samples"):
        ex = hf_dataset[i]
        if not get_article_text(ex):
            continue
        if get_binary_label_fake_is_one(ex) is None:
            continue
        if config.require_image_source and not row_has_image_source(ex):
            continue
        valid.append(i)
    return valid

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════
def train_fold(model, train_loader, val_loader, test_loader, device, fold_idx, cw,
               val_tfidf_probs=None, test_tfidf_probs=None) -> dict:

    use_cuda = device.type == "cuda"
    autocast_ctx, scaler = build_amp_helpers(use_cuda and config.use_amp)

    model.freeze_backbones()
    print(f"  Phase 1: backbones frozen for {config.freeze_epochs} epochs")

    param_groups = model.head_params_for_optimizer(config.lr_head) + \
                   model.backbone_params_for_optimizer(config.lr_backbone)
    optimizer = torch.optim.AdamW(param_groups)

    steps_ep = max(1, len(train_loader) // config.grad_accumulation)
    t_total = steps_ep * config.epochs
    warmup = max(1, int(config.warmup_ratio * t_total))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup, num_training_steps=t_total
    )

    best_val_f1 = -1.0
    best_snapshot = None
    no_improve = 0
    epoch_logs = []

    for epoch in range(1, config.epochs + 1):
        if epoch == config.freeze_epochs + 1:
            model.unfreeze_last_layers(config.unfreeze_last_layers)
            print(f"\n  Phase 2: last {config.unfreeze_last_layers} layers unfrozen")
            new_bb_params = model.backbone_params_for_optimizer(config.lr_backbone)
            optimizer.param_groups[-1]["params"] = new_bb_params[0]["params"]

        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_preds, train_labels_ep = [], []

        pbar = tqdm(train_loader,
                   desc=f"Fold {fold_idx+1} | Ep {epoch:>2}/{config.epochs}",
                   leave=False)
        for step, batch in enumerate(pbar):
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                     for k, v in batch.items()}
            if "metadata" in batch:
                batch["metadata"] = {k: v.to(device) for k, v in batch["metadata"].items()}

            with autocast_ctx():
                out = model(**batch, labels=labels, class_weights=cw)
                loss = out["loss"] / config.grad_accumulation

            if use_cuda:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % config.grad_accumulation == 0 or (step + 1) == len(train_loader):
                if use_cuda:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                if use_cuda:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            train_preds.extend(out["logits"].argmax(-1).detach().cpu().tolist())
            train_labels_ep.extend(labels.cpu().tolist())
            pbar.set_postfix(loss=f"{loss.item()*config.grad_accumulation:.4f}")

        # Evaluate
        tr_acc = accuracy_score(train_labels_ep, train_preds)
        y_val, val_neural_probs = collect_probabilities(model, val_loader, device, autocast_ctx, cw)
        y_te, test_neural_probs = collect_probabilities(model, test_loader, device, autocast_ctx, cw)
        
        blend_w, threshold, m_val, p_val = tune_binary_blend(
            y_val, val_neural_probs, val_tfidf_probs)
        test_blend = (
            test_neural_probs if test_tfidf_probs is None
            else blend_w * test_neural_probs + (1.0 - blend_w) * test_tfidf_probs
        )
        p_te = probs_to_preds(test_blend, threshold)
        m_test = compute_metrics(y_te, p_te)
        gap = tr_acc - m_val["accuracy"]
        
        epoch_logs.append({
            "fold": fold_idx + 1,
            "epoch": epoch,
            "train_acc": tr_acc,
            "val": m_val,
            "test": m_test,
            "gap": gap,
            "blend_w": blend_w,
            "threshold": threshold
        })

        print(
            f"Fold {fold_idx+1} | Ep {epoch:>2}/{config.epochs} | "
            f"TrainAcc={tr_acc:.4f} | "
            f"VAL acc={m_val['accuracy']:.4f} prec={m_val['precision']:.4f} "
            f"rec={m_val['recall']:.4f} f1={m_val['f1']:.4f} | "
            f"TEST acc={m_test['accuracy']:.4f} prec={m_test['precision']:.4f} "
            f"rec={m_test['recall']:.4f} f1={m_test['f1']:.4f} | "
            f"Gap={gap:+.4f} | blend={blend_w:.2f} thr={threshold:.2f}"
        )

        if m_val["f1"] > best_val_f1:
            best_val_f1 = m_val["f1"]
            best_snapshot = {
                "epoch": epoch,
                "val": dict(m_val),
                "test": dict(m_test),
                "gap": gap,
                "blend_w": blend_w,
                "threshold": threshold,
            }
            no_improve = 0
            
            if config.save_checkpoints:
                os.makedirs(config.checkpoint_dir, exist_ok=True)
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "metrics": best_snapshot,
                }, os.path.join(config.checkpoint_dir, f"fold{fold_idx+1}_best.pt"))
        else:
            if epoch > max(config.freeze_epochs,
                          int(config.warmup_ratio * config.epochs) + 1):
                no_improve += 1
            if no_improve >= config.early_stopping_patience:
                print(f"  Early stopping at epoch {epoch}")
                break

    assert best_snapshot is not None
    bv, bt = best_snapshot["val"], best_snapshot["test"]
    print(
        f"\n--- Fold {fold_idx+1} best (val F1 @ epoch {best_snapshot['epoch']}) ---\n"
        f"  Val : acc={bv['accuracy']:.4f} prec={bv['precision']:.4f} "
        f"rec={bv['recall']:.4f} f1={bv['f1']:.4f}\n"
        f"  Test: acc={bt['accuracy']:.4f} prec={bt['precision']:.4f} "
        f"rec={bt['recall']:.4f} f1={bt['f1']:.4f}\n"
    )
    return {"fold": fold_idx + 1, "best_by_val_f1": best_snapshot, "epoch_logs": epoch_logs}

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  RESULTS
# ══════════════════════════════════════════════════════════════════════
def save_and_print_results(fold_reports: list[dict]):
    os.makedirs(config.results_dir, exist_ok=True)
    path_best = os.path.join(config.results_dir, f"{config.results_file_prefix}_results.csv")

    keys = ["fold", "best_epoch", "val_accuracy", "val_precision", "val_recall", "val_f1",
            "test_accuracy", "test_precision", "test_recall", "test_f1",
            "overfit_gap", "blend_w", "threshold"]
    
    best_rows = []
    for fr in fold_reports:
        b = fr["best_by_val_f1"]
        best_rows.append({
            "fold": fr["fold"],
            "best_epoch": b["epoch"],
            "val_accuracy": b["val"]["accuracy"],
            "val_precision": b["val"]["precision"],
            "val_recall": b["val"]["recall"],
            "val_f1": b["val"]["f1"],
            "test_accuracy": b["test"]["accuracy"],
            "test_precision": b["test"]["precision"],
            "test_recall": b["test"]["recall"],
            "test_f1": b["test"]["f1"],
            "overfit_gap": b.get("gap", 0.0),
            "blend_w": b.get("blend_w", 1.0),
            "threshold": b.get("threshold", 0.5),
        })

    mean_row = {"fold": "MEAN", "best_epoch": ""}
    std_row = {"fold": "STD", "best_epoch": ""}
    for k in keys[2:]:
        vals = [r[k] for r in best_rows]
        mean_row[k] = float(np.mean(vals))
        std_row[k] = float(np.std(vals))

    with open(path_best, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in best_rows:
            w.writerow(row)
        w.writerow(mean_row)
        w.writerow(std_row)

    print("\n" + "═" * 100)
    print("RESULTS — ViLT + CLIP Dual Fusion — FakeNewsNet Binary")
    print("═" * 100)
    
    for row in best_rows:
        print(
            f"  Fold {row['fold']}: ep={row['best_epoch']:>2} | "
            f"VAL acc={row['val_accuracy']:.4f} f1={row['val_f1']:.4f} | "
            f"TEST acc={row['test_accuracy']:.4f} "
            f"prec={row['test_precision']:.4f} "
            f"rec={row['test_recall']:.4f} "
            f"f1={row['test_f1']:.4f}"
        )
    
    print("\nMean Test Metrics:")
    for k in ["accuracy", "precision", "recall", "f1"]:
        vals = [r[f"test_{k}"] for r in best_rows]
        print(f"  {k:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
    
    ma = mean_row["test_accuracy"]
    print(f"\n  {'✅ TARGET MET' if ma >= 0.89 else f'⚠️ Gap to 89%: {0.89-ma:.4f}'}")
    print(f"\n  Results saved: {path_best}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  MAIN EXECUTION
# ══════════════════════════════════════════════════════════════════════
def main():
    from datasets import load_dataset
    
    device = resolve_device(force_cpu=config.force_cpu)
    print(f"\n{'='*72}")
    print(f"  ViLT + CLIP Dual Fusion — FakeNewsNet")
    print(f"{'='*72}")
    print(f"  Device: {device}")
    if device.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  ViLT: {config.vilt_model_name}")
    print(f"  CLIP: {config.clip_model_name}")
    print(f"  Metadata: {config.use_metadata}")
    print(f"  Epochs: {config.epochs} | CV: {config.n_splits}-fold")
    print(f"{'='*72}\n")

    print("Loading processors…")
    vilt_processor = ViltProcessor.from_pretrained(config.vilt_model_name)
    clip_processor = CLIPProcessor.from_pretrained(config.clip_model_name)

    print("\nLoading FakeNewsNet…")
    ds = load_dataset(config.dataset_name, split=config.dataset_split)
    n = len(ds) if config.max_samples is None else min(config.max_samples, len(ds))
    ds = ds.select(range(n))
    print(f"  Rows: {len(ds)}")

    valid_idx = build_valid_indices(ds, len(ds))
    print(f"  Valid: {len(valid_idx)}/{len(ds)}")

    labels_all = [get_binary_label_fake_is_one(ds[i]) for i in valid_idx]
    y_all = np.array(labels_all, dtype=np.int64)
    counts = np.bincount(y_all, minlength=2)
    print(f"  Classes: real={counts[0]} | fake={counts[1]}\n")

    # Stratified split
    pos = np.arange(len(valid_idx))
    pos_train, pos_temp, y_tr, y_tmp = train_test_split(
        pos, y_all, test_size=(config.val_frac + config.test_frac),
        stratify=y_all, random_state=SEED
    )
    rel_test = config.test_frac / (config.val_frac + config.test_frac)
    pos_val, pos_test, _, _ = train_test_split(
        pos_temp, y_tmp, test_size=rel_test, stratify=y_tmp, random_state=SEED
    )

    abs_train_pool = [valid_idx[int(i)] for i in pos_train]
    abs_val = [valid_idx[int(i)] for i in pos_val]
    abs_test = [valid_idx[int(i)] for i in pos_test]
    y_train_pool = y_tr

    print(f"Split: train={len(abs_train_pool)} | val={len(abs_val)} | test={len(abs_test)}\n")

    # Metadata encoder
    metadata_encoder = MetadataEncoder(max_domains=config.max_domains)
    metadata_encoder.fit(ds, abs_train_pool)

    # Create datasets
    val_ds = DualEncoderDataset(ds, vilt_processor, clip_processor, metadata_encoder, abs_val)
    test_ds = DualEncoderDataset(ds, vilt_processor, clip_processor, metadata_encoder, abs_test)
    val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False,
                            num_workers=config.num_workers, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False,
                             num_workers=config.num_workers, collate_fn=collate_fn)

    skf = StratifiedKFold(n_splits=config.n_splits, shuffle=True, random_state=SEED)
    fold_reports = []

    for fold_idx, (tr_rel, _) in enumerate(skf.split(np.zeros(len(abs_train_pool)), y_train_pool)):
        print(f"\n{'═'*72}")
        print(f"  FOLD {fold_idx+1} / {config.n_splits}")
        print(f"{'═'*72}")

        fold_train_abs = [abs_train_pool[i] for i in tr_rel]
        fold_labels = np.array(
            [get_binary_label_fake_is_one(ds[i]) for i in fold_train_abs], dtype=np.int64
        )
        cw = class_weights_from_labels(fold_labels, device)
        print(f"  Train: {len(fold_train_abs)} | weights [real={cw[0]:.3f}, fake={cw[1]:.3f}]")

        # TF-IDF ensemble
        val_tfidf_probs, test_tfidf_probs = None, None
        if config.use_tfidf_ensemble:
            vec, tfidf_clf, tfidf_train_acc = build_tfidf_model(ds, fold_train_abs)
            val_tfidf_probs = tfidf_predict_proba(vec, tfidf_clf, ds, abs_val)
            test_tfidf_probs = tfidf_predict_proba(vec, tfidf_clf, ds, abs_test)
            tfidf_val_pred = probs_to_preds(val_tfidf_probs)
            tfidf_val_metrics = compute_metrics(y_all[pos_val], tfidf_val_pred)
            print(
                f"  TF-IDF: train_acc={tfidf_train_acc:.4f} "
                f"val_acc={tfidf_val_metrics['accuracy']:.4f} "
                f"val_f1={tfidf_val_metrics['f1']:.4f}"
            )

        train_ds = DualEncoderDataset(ds, vilt_processor, clip_processor,
                                      metadata_encoder, fold_train_abs)
        train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                                  num_workers=config.num_workers, collate_fn=collate_fn)

        model = ViLTCLIPDualFusion(
            num_domains=metadata_encoder.num_domains,
            num_classes=config.num_classes
        ).to(device)
        
        report = train_fold(
            model, train_loader, val_loader, test_loader, device, fold_idx, cw,
            val_tfidf_probs=val_tfidf_probs, test_tfidf_probs=test_tfidf_probs
        )
        fold_reports.append(report)
        
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Summary
    print(f"\n{'='*72}")
    print(f"  SUMMARY — {config.n_splits}-Fold CV")
    print(f"{'='*72}")
    for fr in fold_reports:
        b = fr["best_by_val_f1"]
        print(f"  Fold {fr['fold']}: ep={b['epoch']:>2} | TEST acc={b['test']['accuracy']:.4f} f1={b['test']['f1']:.4f}")

    metric_keys = ["accuracy", "precision", "recall", "f1"]
    print()
    for k in metric_keys:
        vals = [fr["best_by_val_f1"]["test"][k] for fr in fold_reports]
        print(f"  Mean Test {k:10s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

    save_and_print_results(fold_reports)


if __name__ == "__main__":
    main()

In [ ]:
# Run the training
main()